# Task 03 — Feature Engineering and Hyperparameter Tuning

This notebook improves a baseline classifier on the Breast Cancer Wisconsin dataset.

**Includes**
- Three engineered features derived from existing measurements
- 5-fold stratified cross-validation
- GridSearchCV hyperparameter tuning
- Baseline vs tuned-model comparison using Accuracy, Precision, Recall, F1, and ROC-AUC

The final test set is kept separate from cross-validation and tuning.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.base import clone

RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target  # sklearn: 0 = malignant, 1 = benign

print("Dataset shape:", X.shape)
print("Target names:", data.target_names)
print("Class counts:\n", y.map({0: "malignant", 1: "benign"}).value_counts())

## 1. Feature engineering

Create three additional features from existing measurements:
- `radius_area_proxy`: mean radius × mean perimeter
- `texture_smoothness_interaction`: mean texture × mean smoothness
- `worst_to_mean_radius_ratio`: worst radius divided by mean radius

These are simple, interpretable transformations; their value is assessed using held-out scores.

In [ ]:
def add_engineered_features(frame):
    frame = frame.copy()
    frame["radius_area_proxy"] = frame["mean radius"] * frame["mean perimeter"]
    frame["texture_smoothness_interaction"] = frame["mean texture"] * frame["mean smoothness"]
    frame["worst_to_mean_radius_ratio"] = frame["worst radius"] / (frame["mean radius"] + 1e-8)
    return frame

X_engineered = add_engineered_features(X)
print("Original feature count:", X.shape[1])
print("Engineered feature count:", X_engineered.shape[1])
X_engineered[["radius_area_proxy", "texture_smoothness_interaction", "worst_to_mean_radius_ratio"]].head()

## 2. Train/test split and evaluation setup

Use a stratified 80/20 split. Cross-validation and hyperparameter search use only the training set. The test set is used once for the final before/after comparison.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_engineered, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def score_model(model, X_eval, y_eval):
    model.fit(X_train, y_train)
    pred = model.predict(X_eval)
    # Positive class is malignant (label 0), so use explicit labels for clinical interpretation.
    # sklearn's target is 0=malignant and 1=benign.
    malignant_precision = precision_score(y_eval, pred, pos_label=0, zero_division=0)
    malignant_recall = recall_score(y_eval, pred, pos_label=0, zero_division=0)
    malignant_f1 = f1_score(y_eval, pred, pos_label=0, zero_division=0)
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_eval)[:, list(model.classes_).index(0)]
        auc = roc_auc_score((y_eval == 0).astype(int), probs)
    else:
        auc = np.nan
    return {
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision (malignant)": malignant_precision,
        "Recall (malignant)": malignant_recall,
        "F1 (malignant)": malignant_f1,
        "ROC-AUC (malignant)": auc,
    }

baseline = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

baseline_cv = cross_validate(
    baseline, X_train, y_train, cv=cv,
    scoring={"accuracy": "accuracy", "f1_malignant": "f1",
             "roc_auc": "roc_auc"},
    return_train_score=False
)
print("Baseline 5-fold CV mean accuracy:", round(baseline_cv["test_accuracy"].mean(), 4))
print("Baseline 5-fold CV mean F1 (default positive label=1):", round(baseline_cv["test_f1_malignant"].mean(), 4))
print("Note: final table below explicitly treats malignant (label 0) as the positive class.")

## 3. Hyperparameter search

GridSearchCV compares Logistic Regression regularization settings using 5-fold cross-validation. The scoring target for tuning is F1 on the malignant class (label 0).

In [ ]:
from sklearn.metrics import make_scorer

malignant_f1_scorer = make_scorer(f1_score, pos_label=0, zero_division=0)

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"],
    "model__class_weight": [None, "balanced"]
}

search = GridSearchCV(
    estimator=baseline,
    param_grid=param_grid,
    scoring=malignant_f1_scorer,
    cv=cv,
    n_jobs=-1,
    refit=True
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best 5-fold CV malignant F1:", round(search.best_score_, 4))
tuned_model = search.best_estimator_

## 4. Before-and-after test scores

Compare the baseline pipeline with the tuned pipeline on the same untouched test set. Precision, recall, and F1 explicitly use malignant as the positive class.

In [ ]:
baseline_fitted = clone(baseline).fit(X_train, y_train)
baseline_scores = score_model(baseline_fitted, X_test, y_test)
tuned_scores = score_model(tuned_model, X_test, y_test)

comparison = pd.DataFrame(
    [baseline_scores, tuned_scores],
    index=["Before tuning (baseline)", "After tuning + engineered features"]
).T
comparison["Change"] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
display(comparison.round(4))

print("\nBest model by malignant-class F1:",
      "After tuning" if tuned_scores["F1 (malignant)"] > baseline_scores["F1 (malignant)"] else
      "Baseline (tuning did not improve test F1 on this split)")
print("Measured malignant F1 change:",
      round(tuned_scores["F1 (malignant)"] - baseline_scores["F1 (malignant)"], 4))

## 5. Conclusion

Review the comparison table rather than assuming tuning must improve every metric. If the tuned model does not improve the held-out score, report that honestly. The cross-validation score and test score measure different things; the test set should not be used to select parameters.